In [ ]:
# 1) Setup project, pull latest code, and load config
import os
import sys
import logging
from pathlib import Path
import importlib.util

if Path('/content').exists():
    REPO_URL = 'https://github.com/Swapperss/FinanceQAAssistance.git'
    PROJECT_ROOT = Path('/content/FinanceQAAssistance')
    if not PROJECT_ROOT.exists():
        !git clone {REPO_URL} {PROJECT_ROOT}
    else:
        print('Repository already exists, pulling latest changes...')
        %cd {PROJECT_ROOT}
        !git pull
    %cd {PROJECT_ROOT}
else:
    current_path = Path.cwd().resolve()
    PROJECT_ROOT = None
    for candidate_path in [current_path, *current_path.parents]:
        if (candidate_path / 'config' / 'config.py').exists():
            PROJECT_ROOT = candidate_path
            break
    if PROJECT_ROOT is None:
        raise FileNotFoundError('Could not locate config/config.py from current working directory.')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

config_spec = importlib.util.spec_from_file_location('finance_config', PROJECT_ROOT / 'config' / 'config.py')
config_module = importlib.util.module_from_spec(config_spec)
assert config_spec.loader is not None
config_spec.loader.exec_module(config_module)
Config = config_module.Config
config = Config()

BASE_MODEL_ID = config.model_name
ADAPTER_REPO_ID = config.instruction_adapter_repo_id
MAX_NEW_TOKENS = config.instruction_inference_max_new_tokens

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger('instruction_inference')
logger.info('Notebook configuration loaded from config.py')
print('Project root:', PROJECT_ROOT)
print('Base model:', BASE_MODEL_ID)
print('Adapter repo:', ADAPTER_REPO_ID)
print('Max new tokens:', MAX_NEW_TOKENS)

In [ ]:
# 2) Optional Hugging Face login for private repos
from huggingface_hub import login

hf_token = os.getenv('HF_READ_TOKEN', '').strip() or os.getenv('HF_TOKEN', '').strip()
if hf_token:
    login(token=hf_token)
    print('Logged in using HF_READ_TOKEN/HF_TOKEN environment variable.')
else:
    print('HF_READ_TOKEN/HF_TOKEN not found. If repo is private, run:')
    print('from huggingface_hub import notebook_login; notebook_login()')

In [ ]:
# 3) Load tokenizer and model from Hugging Face Hub
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

use_cuda = torch.cuda.is_available()
print('CUDA available:', use_cuda)
if use_cuda:
    print('GPU:', torch.cuda.get_device_name(0))

try:
    tokenizer = AutoTokenizer.from_pretrained(ADAPTER_REPO_ID, use_fast=True)
except Exception:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if use_cuda:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        quantization_config=quant_config,
        device_map='auto',
        trust_remote_code=True,
    )
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

model = PeftModel.from_pretrained(base_model, ADAPTER_REPO_ID)
model.eval()
logger.info('Loaded base model plus instruction LoRA adapter from Hub')
print('Instruction model loaded successfully.')

In [ ]:
# 4) Instruction prompt helper and sample runs
def build_instruction_prompt(instruction, input_text=''):
    instruction = instruction.strip()
    input_text = input_text.strip()
    if input_text:
        return (
            f'### Instruction:\n{instruction}\n\n'
            f'### Input:\n{input_text}\n\n'
            f'### Response:\n'
        )
    return (
        f'### Instruction:\n{instruction}\n\n'
        f'### Response:\n'
    )

def generate_instruction_response(instruction, input_text='', max_new_tokens=MAX_NEW_TOKENS):
    prompt = build_instruction_prompt(instruction, input_text)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=config.instruction_inference_do_sample,
            temperature=config.instruction_inference_temperature,
            top_p=config.instruction_inference_top_p,
            repetition_penalty=config.instruction_inference_repetition_penalty,
            no_repeat_ngram_size=config.instruction_inference_no_repeat_ngram_size,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated_text[len(prompt):].strip()

test_questions = [
    'What should I do if my bank charged an incorrect fee?',
    'How can I dispute an unauthorized charge on my credit card?',
    'What can I do if my payment was posted late because of a bank error?',
]

for idx, question in enumerate(test_questions, start=1):
    print('=' * 90)
    print(f'Q{idx}: {question}')
    print(f'A{idx}: {generate_instruction_response(question)}')